# Significance Testing via Permutation Tests

- Text level was shuffled in training w/ a different random seed for each permutation

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
from src.perm_res import (
    get_avg_r2,
    get_layer1,
    get_layer2,
    get_layer3,
    get_raw_p_df,
    get_collapsed_null,
    get_corr_p_vals,
)

## Prelim look at baseline 

In [ ]:
parent_base_df = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/parent/pred_df.parquet"
)
self_base_df = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/self/pred_df.parquet"
)

Parent

In [ ]:
r2_df_parent = get_avg_r2(parent_base_df)

In [ ]:
r2_1_parent = get_layer1(r2_df_parent)
print("1 layer:")
display(r2_1_parent)
# 2
r2s_2_S_base_parent, r2s_2_Q_base_parent = get_layer2(r2_df_parent)
print("2 layer S base:")
display(r2s_2_S_base_parent)
print("2 layer Q base:")
display(r2s_2_Q_base_parent)
# 3
r2_3_parent = get_layer3(r2_df_parent)
print("3 layer:")
display(r2_3_parent)

Self

In [ ]:
r2_df_self = get_avg_r2(self_base_df)

In [ ]:
r2_1_self = get_layer1(r2_df_self)
display(r2_1_self)
r2s_2_S_base_self, r2s_2_Q_base_self = get_layer2(r2_df_self)
display(r2s_2_S_base_self)
display(r2s_2_Q_base_self)
r2_3_self = get_layer3(r2_df_self)
display(r2_3_self)

## Aggregate r^2 values across perms

- One table per impact/num-layers combination
- Each table contains r2_S,r2_Q,r2_T for some model configuration

In [ ]:
def create_cmd(num_layer_list, import_dir, export_path, n_threads):
    layer_list_str = " ".join(str(n) for n in num_layer_list)
    cmd_str = f"export PYTHONPATH={BASE_PATH};\
        export OMP_NUM_THREADS={n_threads};\
        uv run python -m src.perm_res\
        --num_layer_list {layer_list_str}\
        --import_dir {import_dir}\
        --export_path {export_path}"
    return " ".join(cmd_str.split())

In [ ]:
perm_check_start = 0
perm_check_end = 1000  # not inclusive
INCLUDE_BASELINE = True

In [ ]:
full_cmd_list = []
loop_dir = BASE_PATH / "perm_res" / "preds"
swarm_dir = BASE_PATH / "swarm_dir/perm_r2"
cmd_dir = swarm_dir / "commands"
log_dir = swarm_dir / "logs"
r2_dir = BASE_PATH / "perm_res" / "r2_tables"
for impact_type in ["parent", "self"]:
    swarm_path = cmd_dir / f"{impact_type}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    if swarm_path.exists():
        swarm_path.unlink()
    cmd_list = []
    for perm_dir in loop_dir.iterdir():
        export_path = r2_dir / perm_dir.name / f"{impact_type}.parquet"
        perm_name = perm_dir.name
        if perm_name == "baseline":
            if not INCLUDE_BASELINE:
                continue
        else:
            perm_num = int(perm_name.split("_")[1].strip(""))
            if (perm_num < perm_check_start) or (perm_num >= perm_check_end):
                print(f"Skipping: {perm_num}")
                continue

        cmd_str = create_cmd(
            num_layer_list=[1, 2, 3],
            import_dir=perm_dir / impact_type,
            n_threads=2,
            export_path=export_path,
        )
        cmd_list.append(cmd_str)
    swarm_path.write_text("\n".join(cmd_list))
    full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run w/ swarm

In [ ]:
from src.swarm import run_perm_r2_swarm

run_perm_r2_swarm(
    impact_list=["parent", "self"],
    log_dir=log_dir,
    cmd_dir=cmd_dir,
    swarm_time="0:15:00",
    n_batch=5,
    partition="quick",
)

Run sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)

## Get p-values

### Raw p-vals for each model config given observed/permutted diffs
- 2 layer: diff = r2_T - r2_S
- 3 layer: diff = r2_T - r2_Q
- NULL distribvution = diff of max(2T) - max(2S/3Q) of each permutation
    - only within hypotheses

In [ ]:
base_imp_path = BASE_PATH / "perm_res" / "r2_tables"
parent_df_list = []
self_df_list = []
for perm_dir in base_imp_path.iterdir():
    self_df = pd.read_parquet(perm_dir / "self.parquet")
    self_df["perm"] = perm_dir.name
    self_df_list.append(self_df)

    parent_df = pd.read_parquet(perm_dir / "parent.parquet")
    parent_df["perm"] = perm_dir.name
    parent_df_list.append(parent_df)

full_r2_self = pd.concat(self_df_list, axis=0).reset_index(drop=True)
full_r2_parent = pd.concat(parent_df_list, axis=0).reset_index(drop=True)

In [ ]:
p_self_2, null_self_2 = get_raw_p_df(num_layers=2, df=full_r2_self, layer_2_base="Q")
p_self_3, null_self_3 = get_raw_p_df(num_layers=3, df=full_r2_self)

p_parent_2, null_parent_2 = get_raw_p_df(
    num_layers=2, df=full_r2_parent, layer_2_base="Q"
)
p_parent_3, null_parent_3 = get_raw_p_df(num_layers=3, df=full_r2_parent)

### Conjuction p-vals

1) Single diff for each model (minimum across hypotheses)
    - Will have to duplicate 2-layer model configs to match 3-model configs

2) Null distribution of diffs (one per permutation)
    - Start w/ 3-dimensional data
        - Axis1: permutations
        - Axis2: model config
        - Axis3: Hypotheses

    - Collapse axis3 by taking minimum diff across all hypotheses
    - Collapse axis2 by taking maximum diff across all model configs
    - End with list of 1001 diffs (including baseline)

In [ ]:
perm_dir_base = BASE_PATH / "perm_res" / "r2_tables"
fill_na_cols = [
    "two_layer_base",
]
keep_cols = [
    "agg_type",
    "llama_model",
    "desc_seg",
    "alpha",
    "r2_S_mean_2",
    "r2_T_mean_2",
    "r2_Q_mean_3",
    "r2_T_mean_3",
    "min_diff",
]
null_dist_parent, parent_baseline = get_collapsed_null(
    impact_type="parent",
    keep_cols=keep_cols,
    fill_na_cols=fill_na_cols,
    perm_dir_base=perm_dir_base,
)
null_dist_self, self_baseline = get_collapsed_null(
    impact_type="self",
    keep_cols=keep_cols,
    fill_na_cols=fill_na_cols,
    perm_dir_base=perm_dir_base,
)

Hypotheses 1/2 comparing 2S/2T + 3Q/3T

In [ ]:
sig_parent_1, non_sig_parent_1 = get_corr_p_vals(
    parent_baseline,
    null_dist_parent,
    title="Null and observed distribution for PARENT conjunction hypotheses",
    diff_col="min_diff",
    show_plot=True,
    xlim=(-0.08, 0.08),
    ylim=(0, 300),
)
sig_self_2, non_sig_self_2 = get_corr_p_vals(
    self_baseline,
    null_dist_self,
    title="Null and observed distribution for SELF conjunction hypotheses",
    diff_col="min_diff",
    show_plot=True,
    xlim=(-0.06, 0.06),
    ylim=(0, 215),
)

Hypothesis 3 comparing 3Q/2T 
- inherently flawed bc this only uses data where 2T is permuted. 
- Answers question of signal(2T) > 0 compared to 3Q baseline, 
    - NOT signal(2T) == signal(3Q) question